# U-Net Segmentation on Oxford-IIIT Pet (Colab)

Self-contained notebook: it downloads the Oxford-IIIT Pet dataset via
`torchvision`, converts its trimap annotations to binary foreground/background
masks in an image/mask layout, and trains, evaluates, predicts, and exports a
U-Net. No imports from `src/` and no CLI calls — every piece of the pipeline
lives in these cells.

- Dataset: `torchvision.datasets.OxfordIIITPet` — 7,349 pet images with pixel-level masks
- Task: binary segmentation (pet foreground vs. background)
- Input: RGB images resized to `CONFIG["data"]["image_size"]`
- Runtime: T4 GPU (`Runtime -> Change runtime type -> T4 GPU`)


## 1. Setup


In [ ]:
import json
import random
import sys
import zipfile
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset

IN_COLAB = "google.colab" in sys.modules
WORK_DIR = Path("/content/segmentation_run") if IN_COLAB else Path.cwd() / "segmentation_run"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("work dir:", WORK_DIR)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())


In [ ]:
%pip install -q torchvision Pillow matplotlib


## 2. Configure the run

The full configuration is defined here, so no YAML file or project checkout is
needed. Lower `train.batch_size` if you hit CUDA out-of-memory errors; raise
`train.epochs` for a longer run.


In [ ]:
CONFIG = {
    "project": {"name": "03-image-segmentation", "seed": 42, "device": "auto"},
    "data": {
        "root": "data",
        "image_dir": "images",
        "mask_dir": "masks",
        "image_size": [128, 128],
        "channels": 3,
        "num_workers": 0,
    },
    "model": {
        "in_channels": 3,
        "out_channels": 1,
        "base_filters": 32,
        "depth": 4,
        "dropout": 0.0,
    },
    "train": {
        "epochs": 20,
        "batch_size": 16,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss": "bce_dice",
        "bce_weight": 0.5,
        "dice_weight": 0.5,
        "scheduler": "plateau",
        "patience": 7,
        "min_delta": 1e-3,
        "checkpoint_dir": "models",
        "best_checkpoint": "best.pt",
        "last_checkpoint": "last.pt",
        "history_path": "results/train_history.json",
    },
    "eval": {
        "batch_size": 16,
        "threshold": 0.5,
        "output_dir": "results/eval",
        "metrics_path": "results/eval/metrics.json",
        "num_overlays": 8,
    },
    "infer": {"threshold": 0.5, "output_dir": "results/predictions"},
}


def path_for(value):
    path = Path(value)
    return path if path.is_absolute() else (WORK_DIR / path).resolve()


print(json.dumps(CONFIG, indent=2))


In [ ]:
def set_seed(seed, deterministic=True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def resolve_device(config):
    requested = str(config["project"].get("device", "auto")).lower()
    if requested != "auto":
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


set_seed(int(CONFIG["project"]["seed"]))
DEVICE = torch.device(resolve_device(CONFIG))
print("device:", DEVICE)


## 3. Download and preprocess Oxford-IIIT Pet

`torchvision.datasets.OxfordIIITPet` provides `trainval` (3,680 pairs) and
`test` (3,669 pairs) with no authentication required. We deterministically split
`trainval` 85%/15% into `train` and `val` using `project.seed`, resize images
and masks to `data.image_size`, and convert the trimap (`1`=pet, `2`=background,
`3`=border) into binary `{0, 255}` masks (`mask == 1`):

```
data/images/{train,val,test}/0000.png
data/masks/{train,val,test}/0000.png
```


In [ ]:
from PIL import Image
from torchvision.datasets import OxfordIIITPet

VAL_FRACTION = 0.15
raw_root = WORK_DIR / "raw_oxford_pet"
data_root = path_for(CONFIG["data"]["root"])
height, width = CONFIG["data"]["image_size"]
image_size = (width, height)

trainval_ds = OxfordIIITPet(
    root=str(raw_root),
    split="trainval",
    target_types="segmentation",
    download=True,
)
test_ds = OxfordIIITPet(
    root=str(raw_root),
    split="test",
    target_types="segmentation",
    download=True,
)

rng = np.random.default_rng(int(CONFIG["project"]["seed"]))
indices = rng.permutation(len(trainval_ds)).tolist()
val_count = int(len(indices) * VAL_FRACTION)
split_sources = {
    "train": (trainval_ds, indices[val_count:]),
    "val": (trainval_ds, indices[:val_count]),
    "test": (test_ds, list(range(len(test_ds)))),
}

for split, (source_ds, split_indices) in split_sources.items():
    image_dir = data_root / CONFIG["data"]["image_dir"] / split
    mask_dir = data_root / CONFIG["data"]["mask_dir"] / split
    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)
    for old_file in list(image_dir.glob("*.png")) + list(mask_dir.glob("*.png")):
        old_file.unlink()

    for out_idx, ds_idx in enumerate(split_indices):
        raw_image, raw_mask = source_ds[ds_idx]
        image = raw_image.convert("RGB").resize(image_size, Image.BILINEAR)
        mask_array = (np.asarray(raw_mask) == 1).astype(np.uint8) * 255
        mask = Image.fromarray(mask_array, mode="L").resize(image_size, Image.NEAREST)
        name = f"{out_idx:04d}.png"
        image.save(image_dir / name)
        mask.save(mask_dir / name)

    print(f"{split}: {len(split_indices)} pairs")


### Inspect a few pairs


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

samples = sorted((data_root / CONFIG["data"]["image_dir"] / "train").glob("*.png"))[:3]
fig, axes = plt.subplots(len(samples), 2, figsize=(8, 3 * len(samples)))
if len(samples) == 1:
    axes = [axes]
for row, image_path in enumerate(samples):
    mask_path = data_root / CONFIG["data"]["mask_dir"] / "train" / image_path.name
    axes[row][0].imshow(Image.open(image_path))
    axes[row][0].set_title("image")
    axes[row][0].axis("off")
    axes[row][1].imshow(Image.open(mask_path), cmap="gray")
    axes[row][1].set_title("mask")
    axes[row][1].axis("off")
plt.tight_layout()
plt.show()


## 4. Dataset and dataloaders

Images and masks share identical filenames and spatial transforms. Images are
converted to `data.channels`, scaled to `[0, 1]`, then normalized to `[-1, 1]`.
Masks are binarized to `{0, 1}` and the training split gets random flips.


In [ ]:
from PIL import Image, ImageOps

SPLITS = ("train", "val", "test")
SUPPORTED_CHANNELS = (1, 3)


class SegmentationDataset(Dataset):
    """Image/mask pairs resized, normalized, and optionally flipped."""

    def __init__(self, pairs, image_size, channels, augment=False):
        if channels not in SUPPORTED_CHANNELS:
            raise ValueError(f"channels must be one of {SUPPORTED_CHANNELS}")
        self.pairs = list(pairs)
        self.image_size = (int(image_size[1]), int(image_size[0]))
        self.channels = channels
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]
        mode = "RGB" if self.channels == 3 else "L"

        with Image.open(image_path) as raw_image:
            image = raw_image.convert(mode).resize(self.image_size, Image.BILINEAR)
        with Image.open(mask_path) as raw_mask:
            mask = raw_mask.convert("L").resize(self.image_size, Image.NEAREST)

        if self.augment:
            if random.random() < 0.5:
                image = ImageOps.mirror(image)
                mask = ImageOps.mirror(mask)
            if random.random() < 0.5:
                image = ImageOps.flip(image)
                mask = ImageOps.flip(mask)

        image_array = np.asarray(image, dtype=np.float32) / 255.0
        mask_array = (np.asarray(mask, dtype=np.float32) > 127.0).astype(np.float32)

        if self.channels == 3:
            image_tensor = torch.from_numpy(image_array).permute(2, 0, 1)
        else:
            image_tensor = torch.from_numpy(image_array).unsqueeze(0)
        image_tensor = (image_tensor - 0.5) / 0.5
        mask_tensor = torch.from_numpy(mask_array).unsqueeze(0)
        return image_tensor, mask_tensor


def list_pairs(config, split):
    if split not in SPLITS:
        raise ValueError(f"unknown split: {split}")
    root = path_for(config["data"]["root"])
    image_dir = root / config["data"]["image_dir"] / split
    mask_dir = root / config["data"]["mask_dir"] / split
    if not image_dir.is_dir():
        raise FileNotFoundError(f"missing image directory: {image_dir}")

    pairs = []
    for image_path in sorted(image_dir.glob("*.png")):
        mask_path = mask_dir / image_path.name
        if not mask_path.is_file():
            raise FileNotFoundError(f"missing mask for {image_path.name}: {mask_path}")
        pairs.append((image_path, mask_path))
    if not pairs:
        raise FileNotFoundError(f"no image/mask pairs found in {image_dir}")
    return pairs


def build_dataloader(config, split, batch_size=None, shuffle=None):
    if split not in SPLITS:
        raise ValueError(f"unknown split: {split}")
    image_size = tuple(int(value) for value in config["data"]["image_size"])
    channels = int(config["data"]["channels"])
    dataset = SegmentationDataset(
        list_pairs(config, split),
        image_size=image_size,
        channels=channels,
        augment=split == "train",
    )
    if batch_size is None:
        batch_size = int(config["train"]["batch_size"])
    if shuffle is None:
        shuffle = split == "train"
    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=shuffle,
        num_workers=int(config["data"].get("num_workers", 0)),
        pin_memory=torch.cuda.is_available(),
    )


def build_dataloaders(config):
    return {split: build_dataloader(config, split) for split in SPLITS}


def image_to_display(image):
    array = image.detach().cpu().float().clamp(-1.0, 1.0)
    array = (array + 1.0) / 2.0
    array = array.permute(1, 2, 0).numpy()
    if array.shape[2] == 1:
        array = np.repeat(array, 3, axis=2)
    return np.clip(array, 0.0, 1.0)


loaders = build_dataloaders(CONFIG)
for split, loader in loaders.items():
    print(f"{split}: {len(loader.dataset)} pairs, {len(loader)} batches")


## 5. Model

U-Net encoder-decoder with concatenative skip connections. It returns per-pixel
logits with shape `(B, 1, H, W)`.


In [ ]:
class ConvBlock(nn.Module):
    """Two 3x3 convolutions with batch norm and ReLU."""

    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """U-Net for binary or multi-class semantic segmentation."""

    def __init__(
        self,
        in_channels=3,
        out_channels=1,
        base_filters=32,
        depth=4,
        dropout=0.0,
    ):
        super().__init__()
        if depth < 1:
            raise ValueError("depth must be >= 1")
        if base_filters <= 0:
            raise ValueError("base_filters must be positive")
        if in_channels <= 0 or out_channels <= 0:
            raise ValueError("in_channels and out_channels must be positive")

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.base_filters = base_filters
        self.depth = depth
        self.dropout = dropout

        self.encoders = nn.ModuleList()
        encoder_channels = []
        channels = in_channels
        for level in range(depth):
            level_channels = base_filters * (2**level)
            self.encoders.append(ConvBlock(channels, level_channels, dropout))
            encoder_channels.append(level_channels)
            channels = level_channels

        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(channels, base_filters * (2**depth), dropout)

        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for level in reversed(range(depth)):
            skip_channels = encoder_channels[level]
            self.upconvs.append(
                nn.ConvTranspose2d(base_filters * (2 ** (level + 1)), skip_channels, 2, stride=2)
            )
            self.decoders.append(ConvBlock(skip_channels * 2, skip_channels, dropout))

        self.head = nn.Conv2d(base_filters, out_channels, 1)

    def forward(self, x):
        if x.dim() != 4:
            raise ValueError(f"expected 4D input (B, C, H, W), got {tuple(x.shape)}")
        if x.shape[1] != self.in_channels:
            raise ValueError(f"expected {self.in_channels} input channels, got {x.shape[1]}")

        factor = 2**self.depth
        height, width = x.shape[-2], x.shape[-1]
        if height % factor != 0 or width % factor != 0:
            raise ValueError(
                f"spatial size {(height, width)} must be divisible by 2**depth={factor}"
            )

        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for upconv, decoder, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = upconv(x)
            x = torch.cat((skip, x), dim=1)
            x = decoder(x)

        return self.head(x)


def build_model(config):
    model = config["model"]
    return UNet(
        in_channels=model["in_channels"],
        out_channels=model["out_channels"],
        base_filters=model["base_filters"],
        depth=model["depth"],
        dropout=model.get("dropout", 0.0),
    )


def load_checkpoint(path, config, device="cpu"):
    model = build_model(config).to(device)
    state = torch.load(path, map_location=device)
    model_state = state.get("model_state", state) if isinstance(state, dict) else state
    model.load_state_dict(model_state)
    model.eval()
    return model


model = build_model(CONFIG)
print(model)


## 6. Losses and metrics

Losses consume raw logits and binary targets. Metrics report the stable keys
`iou`, `dice`, and `pixel_accuracy`.


In [ ]:
def _as_target(logits, targets):
    if not isinstance(logits, torch.Tensor) or not isinstance(targets, torch.Tensor):
        raise TypeError("logits and targets must be torch tensors")

    if targets.shape == logits.shape:
        return targets.to(device=logits.device, dtype=logits.dtype)

    if logits.dim() == 4 and logits.size(1) == 1 and targets.dim() == 3:
        expected = (logits.size(0), logits.size(2), logits.size(3))
        if tuple(targets.shape) == expected:
            return targets.unsqueeze(1).to(device=logits.device, dtype=logits.dtype)

    raise ValueError(
        f"target shape {tuple(targets.shape)} does not match logits shape {tuple(logits.shape)}"
    )


class BCELoss(nn.Module):
    def forward(self, logits, targets):
        return F.binary_cross_entropy_with_logits(logits, _as_target(logits, targets))


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        if smooth <= 0:
            raise ValueError("smooth must be positive")
        self.smooth = smooth

    def forward(self, logits, targets):
        target = _as_target(logits, targets)
        probabilities = torch.sigmoid(logits)
        dims = tuple(range(1, probabilities.dim()))
        intersection = (probabilities * target).sum(dims)
        denominator = probabilities.sum(dims) + target.sum(dims)
        dice = (2 * intersection + self.smooth) / (denominator + self.smooth)
        return 1 - dice.mean()


class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        if bce_weight < 0 or dice_weight < 0:
            raise ValueError("loss weights must be non-negative")
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = BCELoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return self.bce_weight * self.bce(logits, targets) + self.dice_weight * self.dice(
            logits, targets
        )


def build_loss(config):
    train = config["train"]
    name = str(train.get("loss", "bce_dice")).lower()
    if name == "bce":
        return BCELoss()
    if name == "dice":
        return DiceLoss()
    if name in ("bce_dice", "dice_bce", "combined"):
        return CombinedLoss(
            bce_weight=float(train.get("bce_weight", 0.5)),
            dice_weight=float(train.get("dice_weight", 0.5)),
        )
    raise ValueError(f"unknown loss: {name}")


def _metric_target(logits, targets):
    if not isinstance(logits, torch.Tensor) or not isinstance(targets, torch.Tensor):
        raise TypeError("logits and targets must be torch tensors")

    if targets.shape == logits.shape:
        return targets

    if logits.dim() == 4 and logits.size(1) == 1 and targets.dim() == 3:
        expected = (logits.size(0), logits.size(2), logits.size(3))
        if tuple(targets.shape) == expected:
            return targets.unsqueeze(1)

    raise ValueError(
        f"target shape {tuple(targets.shape)} does not match logits shape {tuple(logits.shape)}"
    )


def _prepare(logits, targets, threshold):
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("threshold must be in [0, 1]")
    target = (_metric_target(logits, targets) >= 0.5).float()
    predictions = (torch.sigmoid(logits) >= threshold).float()
    return predictions, target


def iou(logits, targets, threshold=0.5):
    predictions, target = _prepare(logits, targets, threshold)
    intersection = (predictions * target).sum()
    union = predictions.sum() + target.sum() - intersection
    if union.item() == 0:
        return 1.0
    return (intersection / union).item()


def dice(logits, targets, threshold=0.5):
    predictions, target = _prepare(logits, targets, threshold)
    intersection = (predictions * target).sum()
    denominator = predictions.sum() + target.sum()
    if denominator.item() == 0:
        return 1.0
    return (2 * intersection / denominator).item()


def pixel_accuracy(logits, targets, threshold=0.5):
    predictions, target = _prepare(logits, targets, threshold)
    return (predictions == target).float().mean().item()


def compute_metrics(logits, targets, threshold=0.5):
    return {
        "iou": iou(logits, targets, threshold),
        "dice": dice(logits, targets, threshold),
        "pixel_accuracy": pixel_accuracy(logits, targets, threshold),
    }


print("loss:", build_loss(CONFIG))


## 7. Train

Writes `models/best.pt`, `models/last.pt`, and
`results/train_history.json`. CUDA runs use automatic mixed precision.


In [ ]:
def _autocast(device, enabled):
    if enabled:
        return torch.autocast(device_type=device.type, dtype=torch.float16)
    return nullcontext()


def _make_grad_scaler(enabled):
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=enabled)
        except TypeError:
            pass
    return torch.cuda.amp.GradScaler(enabled=enabled)


def _run_epoch(model, loader, loss_fn, device, optimizer=None, scaler=None, use_amp=False, threshold=0.5):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    logits_chunks = []
    target_chunks = []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            with _autocast(device, use_amp):
                logits = model(images)
                loss = loss_fn(logits, targets)

            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        total_loss += loss.item() * images.size(0)
        if not training:
            logits_chunks.append(logits.detach().cpu())
            target_chunks.append(targets.detach().cpu())

    metrics = {"loss": total_loss / max(len(loader.dataset), 1)}
    if not training:
        metrics.update(compute_metrics(torch.cat(logits_chunks), torch.cat(target_chunks), threshold))
    return metrics


def _save_checkpoint(path, model, optimizer, scheduler, epoch, best_val_loss, config, metrics):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
            "best_val_loss": best_val_loss,
            "metrics": metrics,
            "config": config,
        },
        path,
    )


def _load_checkpoint(path, device):
    state = torch.load(path, map_location=device)
    if not isinstance(state, dict) or "model_state" not in state:
        return {"model_state": state, "epoch": -1, "best_val_loss": float("inf")}
    return state


def _build_scheduler(optimizer, train_cfg):
    name = str(train_cfg.get("scheduler", "plateau")).lower()
    if name in ("none", "off", ""):
        return None
    if name == "plateau":
        patience = max(1, int(train_cfg.get("patience", 5)) // 2)
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=patience
        )
    raise ValueError(f"unknown scheduler: {name}")


def train(config, resume=None):
    set_seed(int(config["project"]["seed"]))
    device = torch.device(resolve_device(config))
    model = build_model(config).to(device)
    loss_fn = build_loss(config)
    loaders = build_dataloaders(config)
    train_loader = loaders["train"]
    val_loader = loaders["val"]

    train_cfg = config["train"]
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(train_cfg["lr"]),
        weight_decay=float(train_cfg.get("weight_decay", 0.0)),
    )
    scheduler = _build_scheduler(optimizer, train_cfg)
    use_amp = device.type == "cuda"
    scaler = _make_grad_scaler(use_amp)

    start_epoch = 0
    best_val_loss = float("inf")
    history = []
    if resume is not None:
        state = _load_checkpoint(path_for(resume), device)
        model.load_state_dict(state["model_state"])
        if state.get("optimizer_state") is not None:
            optimizer.load_state_dict(state["optimizer_state"])
        if scheduler is not None and state.get("scheduler_state") is not None:
            scheduler.load_state_dict(state["scheduler_state"])
        start_epoch = int(state.get("epoch", -1)) + 1
        best_val_loss = float(state.get("best_val_loss", best_val_loss))

    checkpoint_dir = path_for(train_cfg["checkpoint_dir"])
    best_path = checkpoint_dir / str(train_cfg["best_checkpoint"])
    last_path = checkpoint_dir / str(train_cfg["last_checkpoint"])
    history_path = path_for(train_cfg["history_path"])
    history_path.parent.mkdir(parents=True, exist_ok=True)

    epochs = int(train_cfg["epochs"])
    patience = int(train_cfg.get("patience", 5))
    min_delta = float(train_cfg.get("min_delta", 0.0))
    threshold = float(config["eval"].get("threshold", 0.5))
    stale_epochs = 0

    for epoch in range(start_epoch, epochs):
        train_metrics = _run_epoch(
            model,
            train_loader,
            loss_fn,
            device,
            optimizer=optimizer,
            scaler=scaler,
            use_amp=use_amp,
        )
        val_metrics = _run_epoch(
            model,
            val_loader,
            loss_fn,
            device,
            use_amp=use_amp,
            threshold=threshold,
        )

        learning_rate = float(optimizer.param_groups[0]["lr"])
        record = {
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "val_loss": val_metrics["loss"],
            "val_iou": val_metrics["iou"],
            "val_dice": val_metrics["dice"],
            "val_pixel_accuracy": val_metrics["pixel_accuracy"],
            "lr": learning_rate,
        }
        history.append(record)
        history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

        improved = val_metrics["loss"] < best_val_loss - min_delta
        if improved:
            best_val_loss = val_metrics["loss"]
            stale_epochs = 0
            _save_checkpoint(
                best_path, model, optimizer, scheduler, epoch, best_val_loss, config, val_metrics
            )
        else:
            stale_epochs += 1

        if scheduler is not None:
            scheduler.step(val_metrics["loss"])

        _save_checkpoint(
            last_path, model, optimizer, scheduler, epoch, best_val_loss, config, val_metrics
        )

        print(
            f"epoch {epoch + 1}/{epochs} "
            f"train_loss={train_metrics['loss']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_iou={val_metrics['iou']:.4f} "
            f"lr={learning_rate:.6f}"
        )

        if stale_epochs >= patience:
            print(f"early stopping after {stale_epochs} stale epoch(s)")
            break

    return {
        "history": history,
        "best_val_loss": best_val_loss,
        "checkpoint_dir": str(checkpoint_dir),
        "history_path": str(history_path),
        "device": str(device),
        "epochs_run": len(history),
    }


result = train(CONFIG)
print("device:", result["device"])
print("epochs run:", result["epochs_run"])
print("best val loss:", f"{result['best_val_loss']:.4f}")
print("history:", result["history_path"])


### Optional: resume from the last checkpoint


In [ ]:
# RESUME = path_for(CONFIG["train"]["checkpoint_dir"]) / CONFIG["train"]["last_checkpoint"]
# result = train(CONFIG, resume=RESUME)


## 8. Evaluate on the test split

Writes `results/eval/metrics.json` and one input/ground-truth/prediction
triptych per overlay.


In [ ]:
from PIL import Image, ImageDraw

COLORS = {"ground_truth": (0, 200, 0), "prediction": (220, 0, 0)}


def _checkpoint_path(config, checkpoint=None):
    if checkpoint is None:
        return path_for(config["train"]["checkpoint_dir"]) / str(
            config["train"]["best_checkpoint"]
        )
    return path_for(checkpoint)


def _to_pil(image):
    return Image.fromarray((np.clip(image, 0.0, 1.0) * 255).astype(np.uint8), mode="RGB")


def _overlay(image, mask, color):
    base = (np.clip(image, 0.0, 1.0) * 255.0).astype(np.float32)
    selected = mask.astype(bool)
    base[selected] = 0.5 * base[selected] + 0.5 * np.asarray(color, dtype=np.float32)
    return Image.fromarray(base.astype(np.uint8), mode="RGB")


def _label(panel, text):
    canvas = Image.new("RGB", (panel.width, panel.height + 20), (255, 255, 255))
    canvas.paste(panel, (0, 20))
    ImageDraw.Draw(canvas).text((4, 4), text, fill=(0, 0, 0))
    return canvas


def _save_triplet(image, target, prediction, path):
    display = image_to_display(image)
    target_mask = target.squeeze(0).numpy()
    prediction_mask = prediction.squeeze(0).numpy()
    panels = [
        _label(_to_pil(display), "input"),
        _label(_overlay(display, target_mask, COLORS["ground_truth"]), "ground truth"),
        _label(_overlay(display, prediction_mask, COLORS["prediction"]), "prediction"),
    ]

    width = sum(panel.width for panel in panels)
    height = max(panel.height for panel in panels)
    canvas = Image.new("RGB", (width, height), (255, 255, 255))
    offset = 0
    for panel in panels:
        canvas.paste(panel, (offset, 0))
        offset += panel.width
    canvas.save(path)


def evaluate(config, checkpoint=None, split="test"):
    if split not in SPLITS:
        raise ValueError(f"unknown split: {split}")

    device = torch.device(resolve_device(config))
    checkpoint_path = _checkpoint_path(config, checkpoint)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"checkpoint not found: {checkpoint_path}")

    model = load_checkpoint(checkpoint_path, config, device)
    loader = build_dataloader(
        config, split, batch_size=int(config["eval"]["batch_size"]), shuffle=False
    )
    threshold = float(config["eval"].get("threshold", 0.5))

    image_chunks = []
    target_chunks = []
    logit_chunks = []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)
            logits = model(images)
            image_chunks.append(images.cpu())
            target_chunks.append(targets.cpu())
            logit_chunks.append(logits.cpu())

    all_images = torch.cat(image_chunks)
    all_targets = torch.cat(target_chunks)
    all_logits = torch.cat(logit_chunks)
    metrics = compute_metrics(all_logits, all_targets, threshold)

    output_dir = path_for(config["eval"]["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions = (torch.sigmoid(all_logits) >= threshold).float()
    dataset = loader.dataset
    overlay_count = min(int(config["eval"].get("num_overlays", 8)), len(dataset))
    overlay_paths = []
    for index in range(overlay_count):
        stem = dataset.pairs[index][0].stem
        overlay_path = output_dir / f"{stem}_overlay.png"
        _save_triplet(all_images[index], all_targets[index], predictions[index], overlay_path)
        overlay_paths.append(str(overlay_path))

    metrics_path = path_for(config["eval"]["metrics_path"])
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "split": split,
        "checkpoint": str(checkpoint_path),
        "num_samples": len(dataset),
        **metrics,
    }
    metrics_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    print(f"split: {split}")
    print(f"samples: {len(dataset)}")
    print(f"iou: {metrics['iou']:.4f}")
    print(f"dice: {metrics['dice']:.4f}")
    print(f"pixel_accuracy: {metrics['pixel_accuracy']:.4f}")
    print(f"metrics: {metrics_path}")
    return {**payload, "metrics_path": str(metrics_path), "overlays": overlay_paths}


metrics = evaluate(CONFIG, split="test")


## 9. Inspect results


In [ ]:
metrics_path = path_for(CONFIG["eval"]["metrics_path"])
if metrics_path.is_file():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))

history_path = path_for(CONFIG["train"]["history_path"])
if history_path.is_file():
    history = json.loads(history_path.read_text())
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, [row["train_loss"] for row in history], label="train")
    axes[0].plot(epochs, [row["val_loss"] for row in history], label="val")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[1].plot(epochs, [row["val_iou"] for row in history], label="IoU")
    axes[1].plot(epochs, [row["val_dice"] for row in history], label="Dice")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Validation metrics")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


In [ ]:
from IPython.display import display
from PIL import Image

overlays = sorted(path_for(CONFIG["eval"]["output_dir"]).glob("*_overlay.png"))
print(f"overlays: {len(overlays)}")
for overlay_path in overlays[:4]:
    print(overlay_path.name)
    display(Image.open(overlay_path))


## 10. Predict one test image

Writes `<stem>_mask.png` and `<stem>_overlay.png` into
`results/predictions`.


In [ ]:
def predict_image(config, image_path, checkpoint=None):
    device = torch.device(resolve_device(config))
    checkpoint_path = _checkpoint_path(config, checkpoint)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"checkpoint not found: {checkpoint_path}")

    path = path_for(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"image not found: {path}")

    data = config["data"]
    height, width = (int(value) for value in data["image_size"])
    channels = int(data["channels"])
    mode = "RGB" if channels == 3 else "L"
    with Image.open(path) as raw_image:
        image = raw_image.convert(mode).resize((width, height), Image.BILINEAR)

    array = np.asarray(image, dtype=np.float32) / 255.0
    if channels == 3:
        tensor = torch.from_numpy(array).permute(2, 0, 1)
    else:
        tensor = torch.from_numpy(array).unsqueeze(0)
    tensor = ((tensor - 0.5) / 0.5).unsqueeze(0).to(device)

    model = load_checkpoint(checkpoint_path, config, device)
    threshold = float(config["infer"].get("threshold", 0.5))
    with torch.no_grad():
        logits = model(tensor)
    probabilities = torch.sigmoid(logits)[0, 0].cpu().numpy()
    mask = (probabilities >= threshold).astype(np.uint8) * 255

    output_dir = path_for(config["infer"]["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    mask_path = output_dir / f"{path.stem}_mask.png"
    overlay_path = output_dir / f"{path.stem}_overlay.png"
    Image.fromarray(mask, mode="L").save(mask_path)

    rgb = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    overlay = rgb.copy()
    selected = mask > 0
    overlay[selected] = 0.5 * rgb[selected] + 0.5 * np.asarray([1.0, 0.0, 0.0])
    Image.fromarray((np.clip(overlay, 0.0, 1.0) * 255).astype(np.uint8), mode="RGB").save(
        overlay_path
    )

    print(f"mask: {mask_path}")
    print(f"overlay: {overlay_path}")
    return {"mask": str(mask_path), "overlay": str(overlay_path)}


test_images = sorted(
    (path_for(CONFIG["data"]["root"]) / CONFIG["data"]["image_dir"] / "test").glob("*.png")
)
prediction = predict_image(CONFIG, test_images[0])


## 11. Export

Creates `segmentation_v0.1.0.zip` with weights, history, metrics, overlays,
predictions, and the run configuration.


In [ ]:
run_config_path = WORK_DIR / "config" / "run_config.json"
run_config_path.parent.mkdir(parents=True, exist_ok=True)
run_config_path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

export_path = (
    Path("/content/segmentation_v0.1.0.zip") if IN_COLAB else WORK_DIR / "segmentation_v0.1.0.zip"
)
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
    for pattern in ["models/*.pt", "results/**/*", "config/*.json"]:
        for path in WORK_DIR.glob(pattern):
            if path.is_file():
                zip_file.write(path, path.relative_to(WORK_DIR))
print("export:", export_path, f"{export_path.stat().st_size / 1e6:.1f} MB")

if IN_COLAB:
    from google.colab import files

    files.download(str(export_path))


## Notes

- This notebook is standalone: no `src/` import, no CLI call, no project clone.
- To use a different dataset, write matching image/mask files under
  `data/images/{train,val,test}` and `data/masks/{train,val,test}` and skip
  section 3.
- If CUDA runs out of memory, lower `CONFIG["train"]["batch_size"]` to `8` or `4`.
- `data.image_size` must stay divisible by `2 ** model.depth` (default `16`).
